## Linear Regression

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score


In [2]:
df = pd.read_csv("features_02.csv")
df

,Number,FileName,Valence,Arousal,SocialEngagement,Time,Behavior,Gender,Age,mfcc1_mean,...,chroma6,chroma7,chroma8,chroma9,chroma10,chroma11,chroma12,zcr,rms,spectral_centroid
0,1,Aster_alarmrespondingtohuman_1524,0.86,2.0,2.5,1524,respondingtohuman,M,1.5,-390.488342,...,0.185233,0.209743,0.250518,0.223257,0.470626,0.590350,0.657865,0.311491,0.064246,4358.639553
1,2,Aster_alertoncurtain_1906,-0.50,3.8,-0.5,1906,alert,M,1.5,-433.151917,...,0.281970,0.347086,0.425936,0.432686,0.381827,0.646774,0.564646,0.230179,0.048830,3505.416980
2,3,Aster_aloneresponding_1458,0.23,1.0,1.5,1458,responding,M,1.5,-337.110504,...,0.145383,0.105791,0.186186,0.427773,0.271388,0.222277,0.615558,0.329690,0.086895,4209.269147
3,4,Aster_alonetalking_1335,0.00,1.8,-2.0,1335,selftalk,M,1.5,-417.492706,...,0.283984,0.336734,0.373847,0.252005,0.427924,0.477983,0.482706,0.259977,0.055223,3482.508807
4,5,Aster_angryathomeoccupied_1904,-3.50,4.0,-1.5,1904,angry,M,1.5,-410.802246,...,0.536215,0.519460,0.490378,0.431585,0.375488,0.306628,0.314257,0.231971,0.032142,3584.103029
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,312,Fizzy_scratchingaster_1659,3.00,2.5,4.5,1659,allogroom,F,1.0,-453.329071,...,0.491355,0.461562,0.516075,0.573383,0.490576,0.536580,0.802170,0.107699,0.016766,2115.950399
312,313,Fizzy_sleepy_1200,-0.68,2.0,-1.0,1200,sleepy,F,1.0,-325.722656,...,0.633860,0.584347,0.494283,0.428000,0.399939,0.332801,0.346417,0.210765,0.025693,3482.031936
313,314,Fizzy_talkingrespondingtoherself_1524,0.30,2.0,-1.2,1524,selftalk,F,1.0,-373.357300,...,0.504699,0.528834,0.655875,0.572965,0.394461,0.395497,0.428561,0.387366,0.057758,4816.822092
314,315,Fizzy_talkingtomateandhuman_1323,1.50,2.8,3.2,1323,conversation,F,1.0,-579.687927,...,0.492729,0.771696,0.740513,0.665613,0.457850,0.374878,0.415552,0.173340,0.015644,2604.357468


In [115]:
y = df[["Valence", "Arousal", "SocialEngagement"]]

non_feature_cols = [
    "Number", "FileName",
    "Valence", "Arousal", "SocialEngagement", "Behavior", "Gender"
]

X = df.drop(columns=non_feature_cols)

In [117]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [119]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),    
    ("model", LinearRegression())       
])

In [121]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()), ('model', LinearRegression())])

In [123]:
y_pred = pipeline.predict(X_test)

In [125]:
mae = mean_absolute_error(y_test, y_pred, multioutput="raw_values")
r2 = r2_score(y_test, y_pred, multioutput="raw_values")

for dim, m, r in zip(["Valence", "Arousal", "Social"], mae, r2):
    print(f"{dim}: MAE = {m:.3f}, R² = {r:.3f}")

Valence: MAE = 1.279, R² = -0.061
Arousal: MAE = 0.727, R² = 0.083
Social: MAE = 1.798, R² = -0.068


## KNN

In [129]:
from sklearn.neighbors import KNeighborsRegressor

In [131]:
pipeline2 = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(
        n_neighbors=5,      # 先用 5，很合理
        weights="distance"  # 距离加权，通常更稳
    ))
])


In [133]:
pipeline2.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('knn', KNeighborsRegressor(weights='distance'))])

In [135]:
y_pred2 = pipeline2.predict(X_test)

In [137]:
mae2 = mean_absolute_error(y_test, y_pred2, multioutput="raw_values")
r2_2 = r2_score(y_test, y_pred2, multioutput="raw_values")
for dim, m, r in zip(["Valence", "Arousal", "Social"], mae2, r2_2):
    print(f"{dim}: MAE = {m:.3f}, R² = {r:.3f}")

Valence: MAE = 1.334, R² = -0.278
Arousal: MAE = 0.768, R² = -0.060
Social: MAE = 1.946, R² = -0.205


## Ridge Linear Regression

In [141]:
from sklearn.linear_model import Ridge

In [143]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1)) 
])

In [145]:
for dim in y.columns:
    print(f"--- {dim} ---")
    pipeline.fit(X_train, y_train[dim])
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test[dim], y_pred)
    r2 = r2_score(y_test[dim], y_pred)
    print(f"MAE = {mae:.3f}, R² = {r2:.3f}\n")

--- Valence ---
MAE = 1.273, R² = -0.051

--- Arousal ---
MAE = 0.730, R² = 0.079

--- SocialEngagement ---
MAE = 1.797, R² = -0.064



## Lasso Linear Regression

In [148]:
from sklearn.linear_model import Lasso

In [95]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(alpha=0.1))
])

for dim in y.columns:
    print(f"--- {dim} ---")
    pipeline.fit(X_train, y_train[dim])
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test[dim], y_pred)
    r2 = r2_score(y_test[dim], y_pred)
    print(f"MAE = {mae:.3f}, R² = {r2:.3f}\n")

--- Valence ---
MAE = 1.231, R² = 0.087

--- Arousal ---
MAE = 0.674, R² = 0.091

--- SocialEngagement ---
MAE = 1.810, R² = -0.016



## Decision Tree

In [98]:
from sklearn.tree import DecisionTreeRegressor

In [150]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),   
    ("model", DecisionTreeRegressor(
        max_depth=None,                
        random_state= 50
    ))
])

for dim in y.columns:
    print(f"--- {dim} ---")
    pipeline.fit(X_train, y_train[dim])
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test[dim], y_pred)
    r2 = r2_score(y_test[dim], y_pred)
    print(f"MAE = {mae:.3f}, R² = {r2:.3f}\n")

--- Valence ---
MAE = 1.663, R² = -1.050

--- Arousal ---
MAE = 0.998, R² = -0.905

--- SocialEngagement ---
MAE = 2.292, R² = -0.757



## Random Forest

In [154]:
from sklearn.ensemble import RandomForestRegressor

In [156]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),  # RF 对尺度不敏感，可保留一致性
    ("model", RandomForestRegressor(
        n_estimators=100,          # 树的数量，可调整
        max_depth=None,            # 树深度，可调防止过拟合
        random_state=42
    ))
])

# 6. 对每个维度单独训练和预测
for dim in y.columns:
    print(f"--- {dim} ---")
    pipeline.fit(X_train, y_train[dim])
    y_pred = pipeline.predict(X_test)
    mae = mean_absolute_error(y_test[dim], y_pred)
    r2 = r2_score(y_test[dim], y_pred)
    print(f"MAE = {mae:.3f}, R² = {r2:.3f}\n")

--- Valence ---
MAE = 1.176, R² = 0.003

--- Arousal ---
MAE = 0.659, R² = 0.164

--- SocialEngagement ---
MAE = 1.750, R² = 0.009



## Baseline Model Summary & Reflection

## Mini CNN

In [163]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


df = pd.read_csv("features_02.csv")


y = df[["Valence", "Arousal", "SocialEngagement"]].values.astype(np.float32)


non_feature_cols = [
    "Number", "FileName",
    "Valence", "Arousal", "SocialEngagement",
    "Time", "Behavior", "Gender", "Age"
]
X = df.drop(columns=non_feature_cols).values.astype(np.float32)


scaler = StandardScaler()
X = scaler.fit_transform(X)

X = X[:, np.newaxis, :]  # shape = (num_samples, 1, num_features)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train)
X_test = torch.tensor(X_test)
y_train = torch.tensor(y_train)
y_test = torch.tensor(y_test)

class AudioDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_dataset = AudioDataset(X_train, y_train)
test_dataset = AudioDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)


class SmallCNN(nn.Module):
    def __init__(self, input_length, output_dim=3):
        super(SmallCNN, self).__init__()
        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool1d(2)
        self.relu = nn.ReLU()
        
        # 计算全连接输入维度
        fc_input_dim = (input_length // 2) * 32  # 因为 MaxPool1d(2)
        self.fc1 = nn.Linear(fc_input_dim, 64)
        self.fc2 = nn.Linear(64, output_dim)
        
    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool(self.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)  # 回归输出
        return x

input_length = X_train.shape[2]
model = SmallCNN(input_length)


criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

for epoch in range(num_epochs):
    model.train()
    train_loss = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * batch_X.size(0)
    train_loss /= len(train_loader.dataset)
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {train_loss:.4f}")

model.eval()
with torch.no_grad():
    y_pred = model(X_test)
    mae = torch.mean(torch.abs(y_pred - y_test), dim=0)
    r2 = 1 - torch.sum((y_test - y_pred)**2, dim=0) / torch.sum((y_test - y_test.mean(dim=0))**2, dim=0)
    
print("\n--- Test Metrics ---")
for i, dim in enumerate(["Valence", "Arousal", "SocialEngagement"]):
    print(f"{dim}: MAE = {mae[i]:.3f}, R² = {r2[i]:.3f}")

ModuleNotFoundError: No module named 'torch'

In [167]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126
   ---------------------------------------- 0.0/2.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/2.6 GB 660.6 kB/s eta 1:05:21
   ---------------------------------------- 0.0/2.6 GB 393.8 kB/s eta 1:49:36
   ---------------------------------------- 0.0/2.6 GB 393.8 kB/s eta 1:49:36
   ---------------------------------------- 0.0/2.6 GB 297.7 kB/s eta 2:25:01
   ---------------------------------------- 0.0/2.6 GB 302.7 kB/s eta 2:22:37
   ---------------------------------------- 0.0/2.6 GB 327.7 kB/s eta 2:11:44
   ---------------------------------------- 0.0/2.6 GB 361.0 kB/s eta 1:59:34
   ---------------------------------------- 0.0/2.6 GB 388.1 kB/s eta 1:51:13
   ---------------------------------------- 0.0/2.6 GB 403.5 kB/s eta 1:46:58
   ---------------------------------------- 0.0/2.6 GB 474.7 kB/s eta 1:30:56
   ---------------------------------------- 0.0/2.6 GB 507.8 kB/s eta 1:25:01
   -----------

In [169]:
import torch
x = torch.rand(5, 3)
print(x)

OSError: [WinError 1114] 动态链接库(DLL)初始化例程失败。 Error loading "C:\Users\hermi\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [171]:
import torch
torch.cuda.is_available()

OSError: [WinError 1114] 动态链接库(DLL)初始化例程失败。 Error loading "C:\Users\hermi\anaconda3\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.